In [27]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn import model_selection

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [28]:
df_adopciones = pd.read_csv("../data/Raw/pet_adoption_data.csv")

### Preparación y limpieza de datos

- En el caso de la variable PetID (Identificador del animal) no es necesaria para la predicción. Por lo que esa variable va a ser eliminada.  
- Creo que para una mejor interpretación y comprensión de los datos en la presentación, es necesario cambiar las edades de meses a años.  
- Para manipular mejor la variable target, voy a cambiarle el nombre de "AdoptionLikelihood" a "target"



In [29]:
# Elimino PetID:

df_adopciones = df_adopciones.drop(columns = ["PetID"], axis = 1)

In [30]:
# Transformación de edad de meses a años:

df_adopciones["AgeYears"] = round(df_adopciones["AgeMonths"]/12, 2)

In [31]:
df_adopciones = df_adopciones.drop(columns = ["AgeMonths"], axis = 1)

In [32]:
# Cambio de nombre de target

df_adopciones = df_adopciones.rename(columns = {"AdoptionLikelihood": "target"})

In [33]:
df_adopciones.duplicated().value_counts()

False    2007
Name: count, dtype: int64

##### Datos limpios:

- Columna PetID eliminada
- Transformación de edad de meses a años
- Cambio de nombre de la variable target
- Dataset sin valores duplicados

El dataset original no tiene valores nulos, por lo que no tengo que realizar ninguna imputación de valores

### Correlación entre variables:

- Variables categóricas: Correladas --> PetType, Breed, Size, Vaccinated, HealthCondition
- Numéricas: Correladas --> AgeYears

Ahora creo dos variables para las features (X_train):

In [34]:
X = df_adopciones.drop(["target"], axis = 1)
y = df_adopciones["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [35]:
features_all = df_adopciones.columns.to_list()

features_corr = df_adopciones.drop(["WeightKg", "TimeInShelterDays", "AdoptionFee", "PreviousOwner", "Color"], axis = 1).columns.to_list()

In [36]:
features_all

['PetType',
 'Breed',
 'Color',
 'Size',
 'WeightKg',
 'Vaccinated',
 'HealthCondition',
 'TimeInShelterDays',
 'AdoptionFee',
 'PreviousOwner',
 'target',
 'AgeYears']

In [37]:
features_corr

['PetType',
 'Breed',
 'Size',
 'Vaccinated',
 'HealthCondition',
 'target',
 'AgeYears']

### Tratamiento de categóricas

In [38]:
# Transformación de las variables: (Sólamente para guardar el dataset limpio)

df_adopciones_all = pd.get_dummies(df_adopciones[features_all], columns = ["PetType", "Breed", "Color", "Size"], dtype = int)
df_adopciones_all

,WeightKg,Vaccinated,HealthCondition,TimeInShelterDays,AdoptionFee,PreviousOwner,target,AgeYears,PetType_Bird,PetType_Cat,...,Breed_Rabbit,Breed_Siamese,Color_Black,Color_Brown,Color_Gray,Color_Orange,Color_White,Size_Large,Size_Medium,Size_Small
0,5.039768,1,0,27,140,0,0,10.92,1,0,...,0,0,0,0,0,1,0,1,0,0
1,16.086727,0,0,8,235,0,0,6.08,0,0,...,1,0,0,0,0,0,1,1,0,0
2,2.076286,0,0,85,385,0,0,11.33,0,0,...,0,0,0,0,0,1,0,0,1,0
3,3.339423,0,0,61,217,1,0,8.08,1,0,...,0,0,0,0,0,0,1,0,0,1
4,20.498100,0,0,28,14,1,0,10.25,0,0,...,1,0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2002,27.039045,1,0,66,26,1,1,6.00,0,0,...,0,0,0,0,0,1,0,0,0,1
2003,4.726954,1,1,59,150,0,0,10.33,0,0,...,1,0,0,1,0,0,0,0,0,1
2004,1.758592,1,0,68,302,0,0,9.42,0,0,...,1,0,0,0,0,1,0,0,0,1
2005,20.961592,1,0,59,478,0,0,1.00,0,0,...,0,0,0,0,1,0,0,1,0,0


In [39]:
df_adopciones_corr = pd.get_dummies(df_adopciones[features_corr], columns = ["PetType", "Breed", "Size"], dtype = int)
df_adopciones_corr

,Vaccinated,HealthCondition,target,AgeYears,PetType_Bird,PetType_Cat,PetType_Dog,PetType_Rabbit,Breed_Golden Retriever,Breed_Labrador,Breed_Parakeet,Breed_Persian,Breed_Poodle,Breed_Rabbit,Breed_Siamese,Size_Large,Size_Medium,Size_Small
0,1,0,0,10.92,1,0,0,0,0,0,1,0,0,0,0,1,0,0
1,0,0,0,6.08,0,0,0,1,0,0,0,0,0,1,0,1,0,0
2,0,0,0,11.33,0,0,1,0,1,0,0,0,0,0,0,0,1,0
3,0,0,0,8.08,1,0,0,0,0,0,1,0,0,0,0,0,0,1
4,0,0,0,10.25,0,0,0,1,0,0,0,0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2002,1,0,1,6.00,0,0,1,0,0,0,0,0,1,0,0,0,0,1
2003,1,1,0,10.33,0,0,0,1,0,0,0,0,0,1,0,0,0,1
2004,1,0,0,9.42,0,0,0,1,0,0,0,0,0,1,0,0,0,1
2005,1,0,0,1.00,0,0,1,0,0,1,0,0,0,0,0,1,0,0


In [40]:
df_adopciones_all.to_csv("../data/Clean/pet_adoption_data_clean_all.csv", index = False)

In [41]:
df_adopciones_corr.to_csv("../data/Clean/pet_adoption_data_clean_corr.csv", index = False)

### DataFrame para las predicciones

In [42]:
df_all = pd.read_csv("../data/Clean/pet_adoption_data_clean_all.csv").drop(columns = ["target"], axis = 1)

In [43]:
df_all.columns

Index(['WeightKg', 'Vaccinated', 'HealthCondition', 'TimeInShelterDays',
       'AdoptionFee', 'PreviousOwner', 'AgeYears', 'PetType_Bird',
       'PetType_Cat', 'PetType_Dog', 'PetType_Rabbit',
       'Breed_Golden Retriever', 'Breed_Labrador', 'Breed_Parakeet',
       'Breed_Persian', 'Breed_Poodle', 'Breed_Rabbit', 'Breed_Siamese',
       'Color_Black', 'Color_Brown', 'Color_Gray', 'Color_Orange',
       'Color_White', 'Size_Large', 'Size_Medium', 'Size_Small'],
      dtype='object')

In [44]:
df_all["TimeInShelterDays"].value_counts()

TimeInShelterDays
15    40
52    36
21    32
6     30
3     30
      ..
40    14
32    14
16    12
80    12
11     9
Name: count, Length: 89, dtype: int64

In [50]:
df_predicciones = pd.DataFrame([
    {
        "WeightKg": 10,
        "Vaccinated": 1,
        "HealthCondition": 0,
        "TimeInShelterDays": 12,
        "AdoptionFee": 100,
        "PreviousOwner": 0,
        "AgeYears": 5,
        "PetType_Bird": 0,
        "PetType_Cat": 0,
        "PetType_Dog": 0,
        "PetType_Rabbit": 1,
        "Breed_Golden Retriever": 0,
        "Breed_Labrador": 0,
        "Breed_Parakeet": 0,
        "Breed_Persian": 0,
        "Breed_Poodle": 0,
        "Breed_Rabbit": 1,
        "Breed_Siamese": 0,
        "Color_Black": 0,
        "Color_Brown": 1,
        "Color_Gray": 0,
        "Color_Orange": 0,
        "Color_White": 0,
        "Size_Large": 0,
        "Size_Medium": 1,
        "Size_Small": 0,
    },
    {
        "WeightKg": 7,
        "Vaccinated": 0,
        "HealthCondition": 1,
        "TimeInShelterDays": 40,
        "AdoptionFee": 250,
        "PreviousOwner": 1,
        "AgeYears": 3,
        "PetType_Bird": 0,
        "PetType_Cat": 1,
        "PetType_Dog": 0,
        "PetType_Rabbit": 0,
        "Breed_Golden Retriever": 0,
        "Breed_Labrador": 0,
        "Breed_Parakeet": 0,
        "Breed_Persian": 0,
        "Breed_Poodle": 0,
        "Breed_Rabbit": 0,
        "Breed_Siamese": 1,
        "Color_Black": 0,
        "Color_Brown": 0,
        "Color_Gray": 0,
        "Color_Orange": 1,
        "Color_White": 0,
        "Size_Large": 0,
        "Size_Medium": 1,
        "Size_Small": 0,
    },
    {
        "WeightKg": 1,
        "Vaccinated": 1,
        "HealthCondition": 1,
        "TimeInShelterDays": 26,
        "AdoptionFee": 70,
        "PreviousOwner": 1,
        "AgeYears": 6,
        "PetType_Bird": 1,
        "PetType_Cat": 0,
        "PetType_Dog": 0,
        "PetType_Rabbit": 0,
        "Breed_Golden Retriever": 0,
        "Breed_Labrador": 0,
        "Breed_Parakeet": 1,
        "Breed_Persian": 0,
        "Breed_Poodle": 0,
        "Breed_Rabbit": 0,
        "Breed_Siamese": 0,
        "Color_Black": 0,
        "Color_Brown": 0,
        "Color_Gray": 1,
        "Color_Orange": 0,
        "Color_White": 0,
        "Size_Large": 0,
        "Size_Medium": 0,
        "Size_Small": 1,
    },
    {
        "WeightKg": 20,
        "Vaccinated": 1,
        "HealthCondition": 1,
        "TimeInShelterDays": 32,
        "AdoptionFee": 300,
        "PreviousOwner": 1,
        "AgeYears": 3,
        "PetType_Bird": 0,
        "PetType_Cat": 1,
        "PetType_Dog": 0,
        "PetType_Rabbit": 0,
        "Breed_Golden Retriever": 0,
        "Breed_Labrador": 0,
        "Breed_Parakeet": 0,
        "Breed_Persian": 1,
        "Breed_Poodle": 0,
        "Breed_Rabbit": 0,
        "Breed_Siamese": 0,
        "Color_Black": 0,
        "Color_Brown": 1,
        "Color_Gray": 0,
        "Color_Orange": 0,
        "Color_White": 0,
        "Size_Large": 1,
        "Size_Medium": 0,
        "Size_Small": 0,
    },
    {
        "WeightKg": 35,
        "Vaccinated": 1,
        "HealthCondition": 1,
        "TimeInShelterDays": 40,
        "AdoptionFee": 120,
        "PreviousOwner": 1,
        "AgeYears": 13,
        "PetType_Bird": 0,
        "PetType_Cat": 0,
        "PetType_Dog": 1,
        "PetType_Rabbit": 0,
        "Breed_Golden Retriever": 0,
        "Breed_Labrador": 1,
        "Breed_Parakeet": 0,
        "Breed_Persian": 0,
        "Breed_Poodle": 0,
        "Breed_Rabbit": 0,
        "Breed_Siamese": 0,
        "Color_Black": 1,
        "Color_Brown": 0,
        "Color_Gray": 0,
        "Color_Orange": 0,
        "Color_White": 0,
        "Size_Large": 1,
        "Size_Medium": 0,
        "Size_Small": 0,   
    },
    {
        "WeightKg": 15,
        "Vaccinated": 1,
        "HealthCondition": 0,
        "TimeInShelterDays": 28,
        "AdoptionFee": 57,
        "PreviousOwner": 0,
        "AgeYears": 10,
        "PetType_Bird": 0,
        "PetType_Cat": 0,
        "PetType_Dog": 0,
        "PetType_Rabbit": 1,
        "Breed_Golden Retriever": 0,
        "Breed_Labrador": 0,
        "Breed_Parakeet": 0,
        "Breed_Persian": 0,
        "Breed_Poodle": 0,
        "Breed_Rabbit": 1,
        "Breed_Siamese": 0,
        "Color_Black": 0,
        "Color_Brown": 0,
        "Color_Gray": 0,
        "Color_Orange": 0,
        "Color_White": 1,
        "Size_Large": 0,
        "Size_Medium": 0,
        "Size_Small": 1,
    },
    {
        "WeightKg": 2,
        "Vaccinated": 1,
        "HealthCondition": 0,
        "TimeInShelterDays": 3,
        "AdoptionFee": 250,
        "PreviousOwner": 0,
        "AgeYears": 0.50,
        "PetType_Bird": 0,
        "PetType_Cat": 1,
        "PetType_Dog": 0,
        "PetType_Rabbit": 0,
        "Breed_Golden Retriever": 0,
        "Breed_Labrador": 0,
        "Breed_Parakeet": 0,
        "Breed_Persian": 1,
        "Breed_Poodle": 0,
        "Breed_Rabbit": 0,
        "Breed_Siamese": 0,
        "Color_Black": 0,
        "Color_Brown": 1,
        "Color_Gray": 0,
        "Color_Orange": 0,
        "Color_White": 0,
        "Size_Large": 1,
        "Size_Medium": 0,
        "Size_Small": 0,
    },
    {
        "WeightKg": 3,
        "Vaccinated": 1,
        "HealthCondition": 1,
        "TimeInShelterDays": 20,
        "AdoptionFee": 232,
        "PreviousOwner": 0,
        "AgeYears": 0.60,
        "PetType_Bird": 0,
        "PetType_Cat": 0,
        "PetType_Dog": 1,
        "PetType_Rabbit": 0,
        "Breed_Golden Retriever": 0,
        "Breed_Labrador": 0,
        "Breed_Parakeet": 0,
        "Breed_Persian": 0,
        "Breed_Poodle": 1,
        "Breed_Rabbit": 0,
        "Breed_Siamese": 0,
        "Color_Black": 0,
        "Color_Brown": 1,
        "Color_Gray": 0,
        "Color_Orange": 0,
        "Color_White": 0,
        "Size_Large": 0,
        "Size_Medium": 1,
        "Size_Small": 0,
    },
    {
        "WeightKg": 45,
        "Vaccinated": 1,
        "HealthCondition": 0,
        "TimeInShelterDays": 50,
        "AdoptionFee": 452,
        "PreviousOwner": 0,
        "AgeYears": 6,
        "PetType_Bird": 0,
        "PetType_Cat": 0,
        "PetType_Dog": 1,
        "PetType_Rabbit": 0,
        "Breed_Golden Retriever": 1,
        "Breed_Labrador": 0,
        "Breed_Parakeet": 0,
        "Breed_Persian": 0,
        "Breed_Poodle": 0,
        "Breed_Rabbit": 0,
        "Breed_Siamese": 0,
        "Color_Black": 0,
        "Color_Brown": 0,
        "Color_Gray": 0,
        "Color_Orange": 1,
        "Color_White": 0,
        "Size_Large": 0,
        "Size_Medium": 0,
        "Size_Small": 1,
    },
    {
        "WeightKg": 27,
        "Vaccinated": 1,
        "HealthCondition": 0,
        "TimeInShelterDays": 45,
        "AdoptionFee": 179,
        "PreviousOwner": 1,
        "AgeYears": 2.5,
        "PetType_Bird": 0,
        "PetType_Cat": 0,
        "PetType_Dog": 1,
        "PetType_Rabbit": 0,
        "Breed_Golden Retriever": 0,
        "Breed_Labrador": 1,
        "Breed_Parakeet": 0,
        "Breed_Persian": 0,
        "Breed_Poodle": 0,
        "Breed_Rabbit": 0,
        "Breed_Siamese": 0,
        "Color_Black": 0,
        "Color_Brown": 1,
        "Color_Gray": 0,
        "Color_Orange": 0,
        "Color_White": 0,
        "Size_Large": 0,
        "Size_Medium": 1,
        "Size_Small": 0,
    }
])

In [51]:
df_predicciones.head()

,WeightKg,Vaccinated,HealthCondition,TimeInShelterDays,AdoptionFee,PreviousOwner,AgeYears,PetType_Bird,PetType_Cat,PetType_Dog,...,Breed_Rabbit,Breed_Siamese,Color_Black,Color_Brown,Color_Gray,Color_Orange,Color_White,Size_Large,Size_Medium,Size_Small
0,10,1,0,12,100,0,5.0,0,0,0,...,1,0,0,1,0,0,0,0,1,0
1,7,0,1,40,250,1,3.0,0,1,0,...,0,1,0,0,0,1,0,0,1,0
2,1,1,1,26,70,1,6.0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
3,20,1,1,32,300,1,3.0,0,1,0,...,0,0,0,1,0,0,0,1,0,0
4,35,1,1,40,120,1,13.0,0,0,1,...,0,0,1,0,0,0,0,1,0,0


In [52]:
df_predicciones.to_csv("../data/Clean/predicciones.csv", index = False)

In [53]:
df_adopciones[features_all].columns

Index(['PetType', 'Breed', 'Color', 'Size', 'WeightKg', 'Vaccinated',
       'HealthCondition', 'TimeInShelterDays', 'AdoptionFee', 'PreviousOwner',
       'target', 'AgeYears'],
      dtype='object')

In [54]:
df_predicciones.columns

Index(['WeightKg', 'Vaccinated', 'HealthCondition', 'TimeInShelterDays',
       'AdoptionFee', 'PreviousOwner', 'AgeYears', 'PetType_Bird',
       'PetType_Cat', 'PetType_Dog', 'PetType_Rabbit',
       'Breed_Golden Retriever', 'Breed_Labrador', 'Breed_Parakeet',
       'Breed_Persian', 'Breed_Poodle', 'Breed_Rabbit', 'Breed_Siamese',
       'Color_Black', 'Color_Brown', 'Color_Gray', 'Color_Orange',
       'Color_White', 'Size_Large', 'Size_Medium', 'Size_Small'],
      dtype='object')